# Chargement cible : BOLELEMI1 uniquement

Copie de `Import_table_baseline.ipynb`, à usage unique, pour ajouter le site
BOLELEMI1 sans recharger les six autres.

**Deux différences avec le notebook de production, et deux seulement :**

1. La cellule `TRUNCATE TABLE` a été retirée. C'est elle qui vide la table et
   impose donc de fournir tous les sites à chaque exécution. Le `MERGE` qui suit
   est en mode `upsert` : il met à jour les lignes existantes et insère les
   manquantes, sans jamais supprimer. Les six autres sites ne sont pas touchés.
2. Le fichier lu est `Baseline_BOLELEMI1.csv` (36 lignes) au lieu de
   `Baseline.csv`.

**Le contrôle de colonnes du notebook de prod est conservé** : si l'en-tête du
fichier ne correspond pas exactement à celui de la table, l'exécution s'arrête.

**À vérifier avant de lancer :**
- les cahiers des charges `2RS` et `IBON` existent dans
  `requirement_specifications`, et leurs identifiants sont bien ceux du CSV ;
- la ligne `BOLELEMI1` existe dans `plants_production_lines`, et son identifiant
  est bien celui du CSV.

Sans ces deux créations, les trois cahiers des charges du fichier porteraient le
même identifiant par défaut et produiraient des doublons de clé métier.

**Après l'exécution**, BOLELEMI1 n'existe que dans la table. La prochaine
exécution du traitement complet vide la table et la reconstruit à partir des
fichiers Excel : BOLELEMI1 en disparaîtrait s'il n'était pas aussi déclaré dans
`Baseline.ipynb`. C'est fait, mais c'est à savoir.

In [0]:
import os
import pandas as pd



import pyspark.sql.utils;
from pyspark.sql.types import StructType, StringType;
from pyspark.sql.functions import concat, lit, col
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType
from datetime import datetime, timedelta
from pyspark.sql.functions import regexp_replace
import numpy as np
from pyspark.sql import SparkSession
#spark.conf.set("spark.sql.execution.arrow.enabled", "true")

In [0]:
try:
    verbose_mode = dbutils.widgets.get("verbose_mode");
except:
    verbose_mode = 'debug'
try:
    current_division = dbutils.widgets.get("division");
except:
    current_division = 'mal'
try:
    current_environment = dbutils.widgets.get("environment");
except:
    current_environment = 'prd'
try:
    current_project = dbutils.widgets.get("project");
except:
    current_project = 'maite_bi'
try:
    current_production_line = dbutils.widgets.get("production_line");
except:
    current_production_line = 'gains'

current_catalog = current_division + '_' + current_project + '_' + current_environment;

current_schema = current_production_line;

current_location = 'abfss://' + current_project + '@adlsdpcom'+ current_environment +f'data.dfs.core.windows.net/' + current_production_line + '/'

if verbose_mode == 'debug':
    display("Debug Mode")
    display(f"current_division : {current_division}")
    display(f"current_environment : {current_environment}")
    display(f"current_project : {current_project}")
    display(f"current_production_line : {current_production_line}")
    display(f"current_catalog : {current_catalog}")
    display(f"current_schema : {current_schema}")
    display(f"current_location : {current_location}")

'Debug Mode'

'current_division : mal'

'current_environment : prd'

'current_project : maite_bi'

'current_production_line : gains'

'current_catalog : mal_maite_bi_prd'

'current_schema : gains'

'current_location : abfss://maite_bi@adlsdpcomprddata.dfs.core.windows.net/gains/'

In [0]:
def delta_table_sql_update_table(source_table, target_table, business_keys, excluded_fields, update_option, verbose_mode):
    """
    Met à jour la table cible avec les données de la table source en excluant les champs exclus et les champs absents de la cible.

    Args:
        source_table: référence de la delta table ou de la vue temporaire source
        target_table: référence de la delta cible
        business_keys: description de la clé de jointure sous la forme [('batch_id',)], ('...',), ('...',)]
        excluded_fields: description des champs à exclure, a minima, les champs d'horodatage [('created_at',), ('updated_at',), ('deleted_at',), ('max_date_time',), ...]
        update_option : mode de mise à jour souhaité ("update" : n'effectue que les mises à jour, "upsert" : effectue les mises à jour et insert, "merge" : effectue les mises à jour et insert et les delete)

    Returns:
    -1 : KO
    0  : No Data from source 
    1 : OK
    """
    function_result = -1
    schema = StructType().add("field_name", StringType())
    # Création des dataframes
    source_df = spark.sql(f"""SELECT * FROM {source_table} LIMIT 1""")
    if source_df.isEmpty():
        function_result = 0
        print("La source est vide")
    else:
        business_keys_df = spark.createDataFrame(business_keys, schema)
        excluded_fields_df = spark.createDataFrame(excluded_fields, schema)

        target_df = spark.sql(f"""SELECT * FROM {target_table} LIMIT 1""")
        target_schema = target_df.schema
        # Champs de la table cible
        target_fields_df = spark.createDataFrame([(c,) for c in target_df.columns], ["field_name"])
        # Typage de la source
        for column_name in source_df.columns:
            if column_name in target_fields_df.columns:
                source_df = source_df.withColumn(column_name, source_df[column_name].cast(target_schema[column_name].dataType)) 
        # Champs de la table source
        source_fields_df = spark.createDataFrame([(c,) for c in source_df.columns], ["field_name"])
        # Champs de la table cible
        target_fields_df = spark.createDataFrame([(c,) for c in target_df.columns], ["field_name"])
        # On supprime les champs absents dans la cible
        common_fields_df = source_fields_df.join(target_fields_df, "field_name", "left_semi")
        # On supprime les champs exclus
        all_fields_df = common_fields_df.join(excluded_fields_df, 'field_name', "left_anti")
        # Champs retenus hors clés
        field_list_df = all_fields_df.join(business_keys_df, 'field_name', "left_anti")
        # Création de la clause SELECT        
        select_clause_df = all_fields_df.withColumn('field_name', col('field_name'))
        select_clause_lst = [row.field_name for row in select_clause_df.collect()]
        select_clause_str = ", \n".join(select_clause_lst)
        # Création de la clause DE JOINTURE
        join_clause_df = business_keys_df.withColumn('field_name', concat(lit("target."), col('field_name'), lit(" = source."), col('field_name')))
        join_clause_lst = [row.field_name for row in join_clause_df.collect()]
        join_clause_str = "\n AND ".join(join_clause_lst)
        # Création de la liste des champs pour l'insertion
        insert_left_clause_df = all_fields_df.withColumn('field_name', col('field_name'))
        insert_left_clause_lst = [row.field_name for row in insert_left_clause_df.collect()]
        insert_left_clause_str = ", \n".join(insert_left_clause_lst)
        insert_left_clause_str += "\n, created_at"
        # Création de la liste des champs à insérer
        insert_right_clause_df = all_fields_df.withColumn('field_name', concat(lit("source."), col('field_name')))
        insert_right_clause_lst = [row.field_name for row in insert_right_clause_df.collect()]
        insert_right_clause_str = ", \n".join(insert_right_clause_lst)
        insert_right_clause_str += "\n, current_timestamp()"
        # Création de la condition pour l'insertion
        update_condition_clause_df = field_list_df.withColumn('field_name', concat(lit("(target."), col('field_name'), lit("<> source."), col('field_name'), lit(") or (target."), col('field_name'), lit(" IS NULL AND source."), col('field_name'), lit(" IS NOT NULL) OR (target."), col('field_name'), lit(" IS NOT NULL AND source."), col('field_name'), lit(" IS NULL)")))
        update_condition_clause_lst = [row.field_name for row in update_condition_clause_df.collect()]
        update_condition_clause_str = "\n OR ".join(update_condition_clause_lst)    #Problème ici
        # Création de la clause d'UPDATE
        update_clause_df = field_list_df.withColumn('field_name', concat(lit("target."), col('field_name'), lit(" = source."), col('field_name')))
        update_clause_lst = [row.field_name for row in update_clause_df.collect()]
        update_clause_str = ", \n ".join(update_clause_lst)
        update_clause_str += "\n, target.updated_at = current_timestamp()"
        #Définition de la requête de MERGE
        merge_query_sql = f"""MERGE INTO 
            {target_table} AS target
        USING
            {source_table} AS source
        ON {join_clause_str}
        WHEN MATCHED AND ({update_condition_clause_str}) 
            THEN UPDATE SET {update_clause_str}"""
        
        if update_option =="upsert" or update_option =="merge" :
            merge_query_sql = merge_query_sql + f"""
        WHEN NOT MATCHED BY TARGET 
            THEN INSERT ({insert_left_clause_str}) 
                VALUES ({insert_right_clause_str})"""
        
        if update_option =="merge" :
            merge_query_sql = merge_query_sql + f"""
        WHEN NOT MATCHED BY SOURCE AND deleted = FALSE
            THEN UPDATE SET
                deleted=TRUE,
                deleted_at = current_timestamp()
            """

        merge_query_sql = merge_query_sql + """
    ;"""

        if verbose_mode == 'debug':
            print('Requête de merge')
            display(merge_query_sql)  

        try:
            spark.sql(merge_query_sql)
            function_result = 1
        except Exception as e:
            print("erreur de merge:", e)
            function_result = -1
    return function_result

In [0]:
current_process="baseline"

In [0]:
configuration_file = f"""/Workspace/Shared/maite-bi/mal-ml-maite-bi/files/configuration/Baseline_BOLELEMI1.csv"""

if not os.path.exists(configuration_file):
    print(f"""Erreur : Le fichier {configuration_file} n'existe pas.""")
    dbutils.notebook.exit(-1)
else:
    print("Check file exists : OK")

Check file exists : OK


In [0]:
configuration_file_df = pd.read_csv(configuration_file, sep=",")

if verbose_mode == 'debug':
    display(configuration_file)
    print( list(configuration_file_df.columns))
    print(configuration_file_df.dtypes)

'/Workspace/Shared/maite-bi/mal-ml-maite-bi/files/configuration/Baseline_BOLELEMI1.csv'

['id_specification', 'specification', 'id_site', 'site', 'Month', 'goods_specy', 'goods_variety', 'nb_individus', 'Baseline_malt_yield_r2', 'Baseline_malt_dry_yield', 'Baseline_goods_weight', 'Baseline_goods_moisture', 'Baseline_malt_moisture', 'Baseline_electricity', 'Baseline_thermal', 'Baseline_FAN', 'Baseline_friabilite', 'Baseline_coloration_EBC', 'Baseline_betaG', 'Baseline_quality', 'created_at', 'updated_at', 'deleted_at']
id_specification             int64
specification               object
id_site                      int64
site                        object
Month                        int64
goods_specy                 object
goods_variety               object
nb_individus                 int64
Baseline_malt_yield_r2     float64
Baseline_malt_dry_yield    float64
Baseline_goods_weight      float64
Baseline_goods_moisture    float64
Baseline_malt_moisture     float64
Baseline_electricity       float64
Baseline_thermal           float64
Baseline_FAN               float64
Basel

In [0]:
target_table_df = spark.sql(f"""SELECT * FROM {current_catalog}.{current_schema}.{current_process} WHERE 1=0""").toPandas()

if verbose_mode == 'debug':
    display(target_table_df.columns)
    print( list(target_table_df.columns))
    print(target_table_df.dtypes)

Index(['id_specification', 'specification', 'id_site', 'site', 'Month',
       'goods_specy', 'goods_variety', 'nb_individus',
       'Baseline_malt_yield_r2', 'Baseline_malt_dry_yield',
       'Baseline_goods_weight', 'Baseline_goods_moisture',
       'Baseline_malt_moisture', 'Baseline_electricity', 'Baseline_thermal',
       'Baseline_FAN', 'Baseline_friabilite', 'Baseline_coloration_EBC',
       'Baseline_betaG', 'Baseline_quality', 'created_at', 'updated_at',
       'deleted_at'],
      dtype='object')

['id_specification', 'specification', 'id_site', 'site', 'Month', 'goods_specy', 'goods_variety', 'nb_individus', 'Baseline_malt_yield_r2', 'Baseline_malt_dry_yield', 'Baseline_goods_weight', 'Baseline_goods_moisture', 'Baseline_malt_moisture', 'Baseline_electricity', 'Baseline_thermal', 'Baseline_FAN', 'Baseline_friabilite', 'Baseline_coloration_EBC', 'Baseline_betaG', 'Baseline_quality', 'created_at', 'updated_at', 'deleted_at']
id_specification                    int32
specification                      object
id_site                             int32
site                               object
Month                               int32
goods_specy                        object
goods_variety                      object
nb_individus                        int32
Baseline_malt_yield_r2            float32
Baseline_malt_dry_yield           float32
Baseline_goods_weight             float32
Baseline_goods_moisture           float32
Baseline_malt_moisture            float32
Baseline_electricit

In [0]:
if not list(sorted(configuration_file_df.columns)) == list(sorted(target_table_df.columns)):
    print("Erreur : Le fichier n'a pas les mêmes colonnes que la table.")
    dbutils.notebook.exit(-1)
else:
    print("Check columns : OK")

Check columns : OK


In [0]:
configuration_file_sdf = spark.createDataFrame(configuration_file_df)

#from pyspark.sql.functions import col, regexp_replace

#columns_to_fix = [
#    "Baseline_malt_yield_r2", "Baseline_malt_dry_yield", "Baseline_goods_weight",
#    "Baseline_goods_moisture", "Baseline_malt_moisture", "Baseline_thermal",
#    "Baseline_gas", "Baseline_heat_pump", "Baseline_coge", "Baseline_electricity"
#]

#for column_name in columns_to_fix:
#    configuration_file_sdf = configuration_file_sdf.withColumn(
#        column_name,
#        regexp_replace(col(column_name), ",", ".").cast("float")
#    )

configuration_file_sdf.createOrReplaceTempView(f"v_ingestion_{current_process}")
if verbose_mode == 'debug' :
    display(configuration_file_sdf)

id_specification,specification,id_site,site,Month,goods_specy,goods_variety,nb_individus,Baseline_malt_yield_r2,Baseline_malt_dry_yield,Baseline_goods_weight,Baseline_goods_moisture,Baseline_malt_moisture,Baseline_electricity,Baseline_thermal,Baseline_FAN,Baseline_friabilite,Baseline_coloration_EBC,Baseline_betaG,Baseline_quality,created_at,updated_at,deleted_at
99,monthly_average,15,BOLELEMI1,1,null,null,44,84.78431818181818,93.12142857142858,286.0554545454546,12.6046511627907,4.23,null,null,151.0,86.7942857142857,3.642142857142857,208.875,null,null,null,null
99,monthly_average,15,BOLELEMI1,2,null,null,32,84.4603125,92.60833333333332,284.3282352941176,12.65882352941176,4.178571428571429,null,null,156.5,70.74142857142857,4.572857142857143,264.3076923076923,null,null,null,null
99,monthly_average,15,BOLELEMI1,3,null,null,63,84.35650793650794,93.16730769230767,284.5224137931035,13.37142857142857,4.299423076923077,null,null,131.1923076923077,75.78423076923077,4.092884615384615,276.9148936170213,null,null,null,null
99,monthly_average,15,BOLELEMI1,4,null,null,53,83.82735849056604,93.152,285.7866037735849,13.92692307692308,4.329607843137255,null,null,128.1764705882353,79.99313725490197,3.902941176470589,247.6875,null,null,null,null
99,monthly_average,15,BOLELEMI1,5,null,null,64,83.8478125,92.90877192982455,285.6014285714286,13.68253968253968,4.245,null,null,128.3914,79.70775862068966,3.7832,238.0178571428571,null,null,null,null
99,monthly_average,15,BOLELEMI1,6,null,null,72,84.04138888888889,92.99473684210524,285.1358571428571,13.42142857142857,4.258983050847458,null,null,136.6296296296296,77.3135593220339,3.946181818181818,238.3684210526316,null,null,null,null
99,monthly_average,15,BOLELEMI1,7,null,null,71,84.14042253521127,93.23174603174604,285.1762121212121,13.83428571428571,4.469206349206349,null,null,136.6190476190476,73.05666666666666,4.02968253968254,262.5849056603773,null,null,null,null
99,monthly_average,15,BOLELEMI1,8,null,null,49,84.2769387755102,93.28222222222222,284.7472222222222,13.73018867924528,4.51,null,null,138.9733333333333,73.10777777777777,4.084222222222222,286.575,null,null,null,null
99,monthly_average,15,BOLELEMI1,9,null,null,45,83.74466666666667,93.11904761904762,285.8588888888889,14.08,4.52,null,null,138.2119047619047,75.87071428571429,4.142857142857143,273.8684210526316,null,null,null,null
99,monthly_average,15,BOLELEMI1,10,null,null,30,84.40599999999999,93.29666666666668,285.6043333333333,13.44666666666667,4.334333333333333,null,null,144.1266666666667,76.51866666666666,4.002,277.0909090909091,null,null,null,null


In [0]:
business_keys = [('id_specification',), ('id_site',), ('Month',)]

In [0]:
excluded_fields = [('created_at',), ('updated_at',), ('deleted_at',)]

In [0]:
data_found = delta_table_sql_update_table(f"v_ingestion_{current_process}", f"{current_catalog}.{current_schema}.{current_process}", business_keys, excluded_fields, "upsert", verbose_mode)
if verbose_mode == 'debug':
    display(f"Merge Result : {data_found}") 
if data_found==-1:
    print("erreur de merge")
    dbutils.notebook.exit(-1)

Requête de merge


'MERGE INTO \n            mal_maite_bi_prd.gains.baseline AS target\n        USING\n            v_ingestion_baseline AS source\n        ON target.id_specification = source.id_specification\n AND target.id_site = source.id_site\n AND target.Month = source.Month\n        WHEN MATCHED AND ((target.specification<> source.specification) or (target.specification IS NULL AND source.specification IS NOT NULL) OR (target.specification IS NOT NULL AND source.specification IS NULL)\n OR (target.site<> source.site) or (target.site IS NULL AND source.site IS NOT NULL) OR (target.site IS NOT NULL AND source.site IS NULL)\n OR (target.goods_specy<> source.goods_specy) or (target.goods_specy IS NULL AND source.goods_specy IS NOT NULL) OR (target.goods_specy IS NOT NULL AND source.goods_specy IS NULL)\n OR (target.goods_variety<> source.goods_variety) or (target.goods_variety IS NULL AND source.goods_variety IS NOT NULL) OR (target.goods_variety IS NOT NULL AND source.goods_variety IS NULL)\n OR (targe

'Merge Result : 1'